<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/pipeline/07_sliding_window_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
BASE_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project"

SPLIT_DIR = os.path.join(BASE_DIR, "data/splits")
WINDOW_DIR = os.path.join(BASE_DIR, "data/windows")

os.makedirs(WINDOW_DIR, exist_ok=True)

WINDOW_SIZE = 60
STEP_SIZE = 10


In [3]:
split_name = "val"   # change to "test" later

ml_file = os.path.join(SPLIT_DIR, f"ml_{split_name}.csv")

ml_df = pd.read_csv(ml_file)
ml_df = ml_df.sort_values("SCLK").reset_index(drop=True)

print("Loaded ML samples:", len(ml_df))


Loaded ML samples: 538525


In [4]:
windows = []
window_id = 0

for start_idx in tqdm(range(0, len(ml_df) - WINDOW_SIZE, STEP_SIZE)):

    end_idx = start_idx + WINDOW_SIZE
    window = ml_df.iloc[start_idx:end_idx].copy()

    right_edge_idx = end_idx - 1

    # Label window based on right edge
    if ml_df.loc[right_edge_idx, "gt_detection_win"] == True:
        label = 1
    else:
        label = 0

    window["window_id"] = window_id
    window["label"] = label

    windows.append(window)
    window_id += 1

print("Total sliding windows:", window_id)


100%|██████████| 53847/53847 [00:39<00:00, 1378.00it/s]

Total sliding windows: 53847


In [5]:
output_file = os.path.join(
    WINDOW_DIR,
    f"{split_name}_sliding_windows_step{STEP_SIZE}.csv"
)

# Clean previous version
if os.path.exists(output_file):
    os.remove(output_file)
    print("Deleted previous file.")

result_df = pd.concat(windows, ignore_index=True)
result_df.to_csv(output_file, index=False)

print("Saved:", output_file)


Saved: /content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows/val_sliding_windows_step10.csv
